# cli

> The central `slmn <tool> [args...]` dispatcher -- one binary, one shell permission, every tool below reachable by name. Tools are plain functions from the other notebooks (`nbtools`, `misc`, `remote`); this notebook just routes argv to them and prints what they return.

In [ ]:
#| default_exp cli

In [ ]:
#| export
import sys, inspect
from fastcore.script import anno_parser
from slmn.nbtools import read_nb, grep_nb, edit_nb, patch_nb_cell, insert_cells
from slmn.misc import read_pdf, gpu_free
from slmn.remote import remote_launch, remote_status, remote_smoke_test, remote_gpu_free, fetch_url, check_ci

In [ ]:
#| export
TOOLS = {
    'read_nb': read_nb,
    'grep_nb': grep_nb,
    'edit_nb': edit_nb,
    'patch_nb_cell': patch_nb_cell,
    'insert_cells': insert_cells,
    'read_pdf': read_pdf,
    'gpu_free': gpu_free,
    'remote_launch': remote_launch,
    'remote_status': remote_status,
    'remote_smoke_test': remote_smoke_test,
    'remote_gpu_free': remote_gpu_free,
    'fetch_url': fetch_url,
    'check_ci': check_ci,
}

In [ ]:
#| export
def _call_cli(fn:callable, argv:list[str]):
    "Parse argv against fn's own signature (reusing fastcore's anno_parser -- the same type-hint-driven parser `call_parse` uses, but invoked directly so we control *when* it runs, since `call_parse` itself auto-executes at decoration time if its module is `__main__`, which doesn't compose with a name-based dispatcher) and call it, returning its result."
    p = anno_parser(fn)
    ns = vars(p.parse_args(argv))
    valid = set(inspect.signature(fn).parameters)
    kwargs = {k: v for k, v in ns.items() if k in valid}
    return fn(**kwargs)

In [ ]:
#| export
def main():
    "Entry point for the `slmn` console script: `slmn <tool> [args...]`. Prints the tool's return value (if any); tools themselves never print."
    if len(sys.argv) < 2 or sys.argv[1] in ('-h', '--help'):
        print("Usage: slmn <tool> [args...]\n\nAvailable tools:")
        for name in sorted(TOOLS): print(f"  {name}")
        sys.exit(0 if len(sys.argv) >= 2 else 1)

    name, argv = sys.argv[1], sys.argv[2:]
    fn = TOOLS.get(name)
    if fn is None:
        print(f"Unknown tool {name!r}. Run `slmn` with no args for the list.", file=sys.stderr)
        sys.exit(1)

    try:
        if name == 'insert_cells':
            # sources is a list[str] -- CLI args don't map cleanly onto that, so read multiple
            # cell sources from stdin instead, separated by a line containing exactly '#---CELL---'.
            if len(argv) < 2:
                print("Usage: slmn insert_cells <path> <anchor_id> < cells.txt", file=sys.stderr)
                sys.exit(1)
            path, anchor_id = argv[0], argv[1]
            sources = sys.stdin.read().split('\n#---CELL---\n')
            result = insert_cells(path, anchor_id, sources)
        else:
            result = _call_cli(fn, argv)
    except SystemExit:
        raise
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)

    if result: print(result)
